In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("ATC-Exploratory-Data-Analysis") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .getOrCreate()
    

:: loading settings :: url = jar:file:/home/ec2-user/.local/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9f8f5fc2-a078-43f2-8e8b-ee65179c7bdd;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 551ms :: artifacts dl 34ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	----------------------------

In [2]:
S3_RAW_ATC = "s3a://de300-project7/raw/atc/Automated_Traffic_Volume_Counts_20260518.csv"

atc = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(S3_RAW_ATC)

print(f"Total Rows Ingested: {atc.count()}")
atc.printSchema()

26/05/20 02:45:22 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Total Rows Ingested: 1875154
root
 |-- RequestID: integer (nullable = true)
 |-- Boro: string (nullable = true)
 |-- Yr: integer (nullable = true)
 |-- M: integer (nullable = true)
 |-- D: integer (nullable = true)
 |-- HH: integer (nullable = true)
 |-- MM: integer (nullable = true)
 |-- Vol: string (nullable = true)
 |-- SegmentID: integer (nullable = true)
 |-- WktGeom: string (nullable = true)
 |-- street: string (nullable = true)
 |-- fromSt: string (nullable = true)
 |-- toSt: string (nullable = true)
 |-- Direction: string (nullable = true)



In [7]:
#rows where 'Vol' contains characters that are NOT digits or minus signs
non_numeric_vol = atc.filter(~F.col("Vol").rlike(r"^-?\d+$"))
print(f"Number of malformed text rows in Vol: {non_numeric_vol.count()}")
non_numeric_vol.select("SegmentID", "Boro", "Yr", "Vol").show(10, truncate=False)

Number of malformed text rows in Vol: 15257
+---------+---------+----+-----+
|SegmentID|Boro     |Yr  |Vol  |
+---------+---------+----+-----+
|136811   |Manhattan|2013|1,147|
|136811   |Manhattan|2013|1,179|
|136811   |Manhattan|2013|1,218|
|136811   |Manhattan|2013|1,199|
|136811   |Manhattan|2013|1,148|
|136811   |Manhattan|2013|1,149|
|136811   |Manhattan|2013|1,192|
|136811   |Manhattan|2013|1,137|
|136811   |Manhattan|2013|1,175|
|136811   |Manhattan|2013|1,184|
+---------+---------+----+-----+
only showing top 10 rows



In [3]:
#total null records across columns
atc.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in atc.columns]).show()

+---------+----+---+---+---+---+---+---+---------+-------+------+------+----+---------+
|RequestID|Boro| Yr|  M|  D| HH| MM|Vol|SegmentID|WktGeom|street|fromSt|toSt|Direction|
+---------+----+---+---+---+---+---+---+---------+-------+------+------+----+---------+
|        0|   0|  0|  0|  0|  0|  0|  0|        0|      0|     0|     0|1246|        0|
+---------+----+---+---+---+---+---+---+---------+-------+------+------+----+---------+



In [4]:
atc.select("Vol").summary("count", "mean", "stddev", "min", "max").show()

print("Negative traffic volumes:", atc.filter(F.col("Vol") < 0).count())
print("Exactly zero traffic volumes:", atc.filter(F.col("Vol") == 0).count())

+-------+------------------+
|summary|               Vol|
+-------+------------------+
|  count|           1875154|
|   mean|103.20391774383205|
| stddev|137.88822155625724|
|    min|                -1|
|    max|               999|
+-------+------------------+



Negative traffic volumes: 1


Exactly zero traffic volumes: 96278


In [8]:
#if 999 acts as an error code by looking at its frequently
print("Rows with exactly 999 trips:", atc.filter(F.col("Vol") == "999").count())

#profile 96k zero-volume rows by hour (HH) to see if they happen at night or all day
print("\nHourly distribution of zero-traffic records:")
atc.filter(F.col("Vol") == "0").groupBy("HH").count().orderBy("HH").show(24)

Rows with exactly 999 trips: 50

Hourly distribution of zero-traffic records:


+---+-----+
| HH|count|
+---+-----+
|  0| 5040|
|  1| 6034|
|  2| 7376|
|  3| 7931|
|  4| 7124|
|  5| 5455|
|  6| 4183|
|  7| 3375|
|  8| 3000|
|  9| 2943|
| 10| 2959|
| 11| 2832|
| 12| 2882|
| 13| 2831|
| 14| 2800|
| 15| 2826|
| 16| 2847|
| 17| 2888|
| 18| 2969|
| 19| 3062|
| 20| 3271|
| 21| 3544|
| 22| 3811|
| 23| 4295|
+---+-----+



In [5]:
#spatial resolution boundaries
print("Unique Traffic Sensor Segments:", atc.select("SegmentID").distinct().count())

print("\nRecord Distribution Across NYC Boroughs:")
atc.groupBy("Boro").count().orderBy(F.desc("count")).show()

Unique Traffic Sensor Segments: 3391

Record Distribution Across NYC Boroughs:


+-------------+------+
|         Boro| count|
+-------------+------+
|       Queens|560928|
|     Brooklyn|540695|
|    Manhattan|343555|
|        Bronx|308124|
|Staten Island|121852|
+-------------+------+



In [6]:
#stitch split integers into unified timestamp
timestamp_test = atc.select("Yr", "M", "D", "HH", "MM").limit(5)

#unified string representations
timestamp_str_expr = F.concat_ws("-", F.col("Yr"), F.col("M"), F.col("D"))
time_str_expr = F.concat_ws(":", F.col("HH"), F.col("MM"), F.lit("00"))
combined_expr = F.concat_ws(" ", timestamp_str_expr, time_str_expr)

#parsed verification layout
timestamp_test.withColumn("parsed_timestamp", F.to_timestamp(combined_expr, "yyyy-M-d H:m:ss")).show()

+----+---+---+---+---+-------------------+
|  Yr|  M|  D| HH| MM|   parsed_timestamp|
+----+---+---+---+---+-------------------+
|2013|  3|  7|  4| 15|2013-03-07 04:15:00|
|2013|  3|  7|  4| 30|2013-03-07 04:30:00|
|2013|  3|  7|  4| 45|2013-03-07 04:45:00|
|2013|  3|  7|  5|  0|2013-03-07 05:00:00|
|2013|  3|  7|  5| 15|2013-03-07 05:15:00|
+----+---+---+---+---+-------------------+



In [9]:
#traffic records count per calendar year
atc.groupBy("Yr").count().orderBy("Yr").show()

+----+------+
|  Yr| count|
+----+------+
|2000|  1904|
|2006|   664|
|2007|  8130|
|2008| 32482|
|2009|122851|
|2010|132016|
|2011|120249|
|2012|128222|
|2013|128762|
|2014|130754|
|2015|124428|
|2016|129617|
|2017|123901|
|2018| 98961|
|2019|115730|
|2020| 40224|
|2021| 85810|
|2022| 78652|
|2023| 85106|
|2024| 95203|
+----+------+
only showing top 20 rows



In [10]:
#rows of data per individual tracking segment
segment_density = atc.groupBy("SegmentID").count()
segment_density.select("count").summary("min", "25%", "50%", "75%", "max").show()

#segments with the lowest data coverage
print("Segments with dangerously low historical rows:")
segment_density.orderBy("count").show(5)

+-------+-----+
|summary|count|
+-------+-----+
|    min|    1|
|    25%|   23|
|    50%|  300|
|    75%|  766|
|    max|13398|
+-------+-----+

Segments with dangerously low historical rows:


+---------+-----+
|SegmentID|count|
+---------+-----+
|   121323|    1|
|   163783|    1|
|   111188|    1|
|   150838|    1|
|   107345|    1|
+---------+-----+
only showing top 5 rows

